# BLEU と ROUGE を定義式から実装する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/B_implementations/bleu_rouge_scratch.ipynb)

NLP クックブック B-23 ／ 仕組みを理解するための実装

機械翻訳と要約の標準的な評価指標である BLEU と ROUGE (ROUGE-1 / ROUGE-2 / ROUGE-L) を定義式から自分で実装し、ライブラリ（`sacrebleu` と `rouge-score`）の出力と完全に一致することを確かめたい。

In [1]:
!pip install -q sacrebleu rouge-score

In [2]:
import math
from collections import Counter
import sacrebleu
from rouge_score import rouge_scorer

def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

def bleu_scratch(hyps_tok, refs_tok, max_n=4):
    """上限クリッピング付き修正 n-gram 適合率と簡潔性ペナルティ (BP) によるコーパス BLEU"""
    clipped_total, cand_total = [0] * max_n, [0] * max_n
    hyp_len = ref_len = 0
    for hyp, refs in zip(hyps_tok, refs_tok):
        hyp_len += len(hyp)
        # 候補長に最も近い参照訳の長さ（同距離なら短いほう）を選ぶ
        ref_len += min((abs(len(r) - len(hyp)), len(r)) for r in refs)[1]
        for n in range(1, max_n + 1):
            h_ng = ngrams(hyp, n)
            max_ref_ng = Counter()
            for r in refs:
                for k, v in ngrams(r, n).items():
                    if v > max_ref_ng[k]:
                        max_ref_ng[k] = v
            clipped_total[n - 1] += sum(min(cnt, max_ref_ng[k]) for k, cnt in h_ng.items())
            cand_total[n - 1] += max(len(hyp) - n + 1, 0)
    precisions = [c / t if t > 0 else 0.0 for c, t in zip(clipped_total, cand_total)]
    bp = 1.0 if hyp_len > ref_len else (math.exp(1.0 - ref_len / hyp_len) if hyp_len > 0 else 0.0)
    if min(precisions) == 0.0:
        return 0.0, precisions, bp
    geo_mean = math.exp(sum(math.log(p) for p in precisions) / max_n)
    return 100.0 * bp * geo_mean, precisions, bp

def lcs_length(a, b):
    """動的計画法で 2 系列の最長共通部分列 (LCS) の長さを求める"""
    dp = [0] * (len(b) + 1)
    for ai in a:
        prev = 0
        for j, bj in enumerate(b, 1):
            cur = dp[j]
            dp[j] = prev + 1 if ai == bj else max(dp[j], dp[j - 1])
            prev = cur
    return dp[-1]

def rouge_n_scratch(ref_tok, hyp_tok, n):
    r_ng, h_ng = ngrams(ref_tok, n), ngrams(hyp_tok, n)
    overlap = sum((r_ng & h_ng).values())
    p = overlap / sum(h_ng.values()) if h_ng else 0.0
    r = overlap / sum(r_ng.values()) if r_ng else 0.0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f

def rouge_l_scratch(ref_tok, hyp_tok):
    lcs = lcs_length(ref_tok, hyp_tok)
    p = lcs / len(hyp_tok) if hyp_tok else 0.0
    r = lcs / len(ref_tok) if ref_tok else 0.0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f

In [3]:
hyps = ["the cat sat on the mat", "he went to tokyo yesterday", "the quick brown fox jumps over the lazy dog"]
refs = ["the cat is sitting on the mat", "yesterday he went to tokyo", "a quick brown fox jumped over a lazy dog"]

mine_bleu, precs, bp = bleu_scratch([h.split() for h in hyps], [[r.split()] for r in refs])
lib_bleu = sacrebleu.corpus_bleu(hyps, [refs], tokenize="none")
print(f"BLEU: 自作={mine_bleu:.6f} / sacrebleu={lib_bleu.score:.6f} / 差={abs(mine_bleu - lib_bleu.score):.2e}")
print("  各 n-gram 適合率:", [round(p * 100, 3) for p in precs], "BP:", round(bp, 4))

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
for i, (h, r) in enumerate(zip(hyps, refs)):
    lib = scorer.score(r, h)
    r1 = rouge_n_scratch(r.split(), h.split(), 1)
    r2 = rouge_n_scratch(r.split(), h.split(), 2)
    rl = rouge_l_scratch(r.split(), h.split())
    print(f"文 {i}: R1={r1[2]:.4f}(差{abs(r1[2] - lib['rouge1'].fmeasure):.1e}) "
          f"R2={r2[2]:.4f}(差{abs(r2[2] - lib['rouge2'].fmeasure):.1e}) "
          f"RL={rl[2]:.4f}(差{abs(rl[2] - lib['rougeL'].fmeasure):.1e})")

BLEU: 自作=30.806403 / sacrebleu=30.806403 / 差=0.00e+00
  各 n-gram 適合率: [80.0, 52.941, 28.571, 9.091] BP: 0.9512
文 0: R1=0.7692(差0.0e+00) R2=0.5455(差0.0e+00) RL=0.7692(差0.0e+00)
文 1: R1=1.0000(差0.0e+00) R2=0.7500(差0.0e+00) RL=0.8000(差0.0e+00)
文 2: R1=0.6667(差0.0e+00) R2=0.3750(差0.0e+00) RL=0.6667(差0.0e+00)


In [4]:
# 定義式の性質を確かめる極端な例
cases = [
    ("単語の重複 (" + "the " * 6 + ")", ["the"] * 6, ["the", "cat", "sat", "on", "the", "mat"]),
    ("短すぎる出力 (BP で減点)", ["the", "cat"], ["the", "cat", "sat", "on", "the", "mat"]),
    ("語順を逆転 (R1=1.0 だが RL が低下)", ["mat", "the", "on", "sat", "cat", "the"], ["the", "cat", "sat", "on", "the", "mat"]),
]
for label, h_tok, r_tok in cases:
    b, _, bp_c = bleu_scratch([h_tok], [[r_tok]])
    r1_f = rouge_n_scratch(r_tok, h_tok, 1)[2]
    rl_f = rouge_l_scratch(r_tok, h_tok)[2]
    print(f"{label:34s} | BLEU={b:5.1f} (BP={bp_c:.3f}) | R1={r1_f:.3f} | RL={rl_f:.3f}")

単語の重複 (the the the the the the )   | BLEU=  0.0 (BP=1.000) | R1=0.333 | RL=0.333
短すぎる出力 (BP で減点)                    | BLEU=  0.0 (BP=0.135) | R1=0.500 | RL=0.500
語順を逆転 (R1=1.0 だが RL が低下)           | BLEU=  0.0 (BP=1.000) | R1=1.000 | RL=0.500


- 上限クリッピング（参照訳に出現する最大回数で頭打ち）がないと `"the the the..."` のような同じ語の繰り返しで適合率が 100% になってしまう。また、短い出力だけで適合率を稼ぐのを防ぐために簡潔性ペナルティ (BP) が掛かる。
- BLEU は 1〜4-gram 適合率の幾何平均をとるため、コーパス全体で 4-gram の一致が 1 件でもゼロになるとスコア全体が 0 になる（文単位の BLEU で平滑化が必要な理由）。
- ROUGE-1 は語順を見ないため語順を逆にしても 1.000 のままになるが、動的計画法による LCS を使う ROUGE-L は語順の崩れを捉えて低下する。実務での利用は、機械翻訳の章の「機械翻訳を BLEU で評価する (sacreBLEU)」と要約と平易化の章の「要約を ROUGE で評価する（日本語対応）」を参照。